# Figure 4: Engineering resilience and system-level robustness
This notebook reproduces Figure 4A (latency breakdown), 4B (Gaussian noise robustness), 4C (packet loss robustness), and 4D (sensor failure tolerance).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Helvetica', 'Arial']
rcParams['axes.linewidth'] = 1.5
rcParams['xtick.major.width'] = 1.5
rcParams['ytick.major.width'] = 1.5

color_trauma = '#E64B35'
color_lstm = '#4DBBD5'
color_red_dark = '#B22222'

data_dir = './data'

In [ ]:
# Panel A: latency components (fixed values)
latency_components = [2.0, 8.0, 15.2, 12.0]
latency_labels = ['Edge\n2.0ms', '5G uplink\n8.0ms', 'Cloud\n15.2ms', 'Downlink\n12.0ms']
latency_colors = ['#4DBBD5', '#00A087', '#E64B35', '#D55E00']
total_latency = sum(latency_components)

# Panel B: noise robustness
df_noise = pd.read_excel(f'{data_dir}/Fig4_Robustness_Noise.xlsx', sheet_name='Fig4_Robustness_Noise')
snr = df_noise['SNR Level'].values
auroc_trauma_noise = df_noise['AUROC TraumaFormer Noise'].values
auroc_lstm_noise = df_noise['AUROC LSTM Noise'].values

# Panel C: packet loss robustness
df_loss = pd.read_csv(f'{data_dir}/Fig4_Robustness_PacketLoss.csv')
miss_rate = df_loss['Missing_Rate_Percent'].values
auroc_trauma_loss = df_loss['AUROC_TraumaFormer'].values
auroc_lstm_loss = df_loss['AUROC_LSTM'].values

# Panel D: sensor failure robustness
df_fault = pd.read_csv(f'{data_dir}/Fig4_Robustness_SensorFailure.csv')
fault_dur = df_fault['Fault_Duration_Seconds'].values
auroc_trauma_fault = df_fault['AUROC_TraumaFormer'].values

In [ ]:
# Create figure
fig = plt.figure(figsize=(14, 10))
gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.25)

# Panel A: stacked bar for latency
ax1 = fig.add_subplot(gs[0, 0])
x = [0]
bottoms = [0]
for i, (val, label, color) in enumerate(zip(latency_components, latency_labels, latency_colors)):
    ax1.bar(x, val, bottom=bottoms, color=color, edgecolor='black', linewidth=1,
            label=label.replace('\n', ' '))
    bottoms = [bottoms[0] + val]
ax1.text(0, total_latency + 2, f'{total_latency:.1f} ms', ha='center', va='bottom',
         fontsize=11, weight='bold', color='black')
ax1.axhline(y=100, color=color_red_dark, linestyle='--', linewidth=2, label='100 ms target')
ax1.set_xticks([])
ax1.set_ylabel('Latency (ms)', fontsize=11)
ax1.set_title('A  End-to-end latency (37.2 ms total)', fontsize=12, loc='left', weight='bold')
ax1.legend(loc='upper right', fontsize=8, frameon=False)
ax1.set_ylim(0, 110)
ax1.grid(axis='y', linestyle='--', alpha=0.5)

# Panel B: Gaussian noise
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(snr, auroc_trauma_noise, color=color_trauma, marker='o', linewidth=2, markersize=6,
         label='Trauma-Former')
ax2.plot(snr, auroc_lstm_noise, color=color_lstm, marker='s', linewidth=2, markersize=6,
         label='LSTM')
ax2.set_xlabel('SNR (dB)', fontsize=11)
ax2.set_ylabel('AUROC', fontsize=11)
ax2.set_title('B  Gaussian noise robustness', fontsize=12, loc='left', weight='bold')
ax2.legend(loc='lower left', fontsize=9, frameon=False)
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.set_ylim(0.7, 1.0)

# Panel C: Packet loss
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(miss_rate, auroc_trauma_loss, color=color_trauma, marker='o', linewidth=2, markersize=6,
         label='Trauma-Former')
ax3.plot(miss_rate, auroc_lstm_loss, color=color_lstm, marker='s', linewidth=2, markersize=6,
         label='LSTM')
ax3.set_xlabel('Missing Rate (%)', fontsize=11)
ax3.set_ylabel('AUROC', fontsize=11)
ax3.set_title('C  Packet loss robustness (imputation-free)', fontsize=12, loc='left', weight='bold')
ax3.legend(loc='lower left', fontsize=9, frameon=False)
ax3.grid(True, linestyle='--', alpha=0.5)
ax3.set_ylim(0.4, 1.0)

# Panel D: Sensor failure
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(fault_dur, auroc_trauma_fault, color=color_trauma, marker='o', linewidth=2, markersize=6,
         label='Trauma-Former')
ax4.set_xlabel('HR Sensor Fault Duration (s)', fontsize=11)
ax4.set_ylabel('AUROC', fontsize=11)
ax4.set_title('D  Transient sensor failure\n(HR missing, rely on BP/SpO₂)', fontsize=12, loc='left', weight='bold')
ax4.legend(loc='lower left', fontsize=9, frameon=False)
ax4.grid(True, linestyle='--', alpha=0.5)
ax4.set_ylim(0.9, 1.0)
ax4.text(15, 0.925, 'AUROC > 0.92\nwith HR missing', ha='center', va='center',
         fontsize=9, bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig('Figure4.tiff', dpi=300, format='tiff', bbox_inches='tight')
plt.show()